# OR · 07 Safety Stock Intro



## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [24]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

✅ Librerías cargadas
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw


## 2️⃣ Cargar y Preparar Datos

In [25]:
# Cargar datos
df_orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=['date'])
df_products = pd.read_csv(DATA_DIR / "products.csv")
df_inventory = pd.read_csv(DATA_DIR / "inventory.csv")

print("📊 Datos cargados:")
print(f"  - Órdenes: {len(df_orders)} registros")
print(f"  - Productos: {len(df_products)} SKUs")
print(f"  - Inventario: {len(df_inventory)} registros")

display(df_orders.head(3))

📊 Datos cargados:
  - Órdenes: 8504 registros
  - Productos: 200 SKUs
  - Inventario: 3000 registros


,order_id,date,sku,qty,location_id,channel
0,ORD-100000,2024-01-01,SKU-00023,13,LOC-013,Retail
1,ORD-100001,2024-01-01,SKU-00111,7,LOC-011,B2B
2,ORD-100002,2024-01-01,SKU-00100,5,LOC-019,Ecom


## 3️⃣ Agregación de Demanda Diaria

In [26]:
# Agregar demanda diaria por SKU
df_daily_demand = df_orders.groupby(['sku', 'date'])['qty'].sum().reset_index()
df_daily_demand.rename(columns={'qty': 'daily_demand'}, inplace=True)

print("📅 Demanda diaria agregada")
print(f"Total registros: {len(df_daily_demand)}")
display(df_daily_demand.head())

# Distribución de demanda para un SKU ejemplo
sample_sku = df_daily_demand['sku'].iloc[0]
sample_data = df_daily_demand[df_daily_demand['sku'] == sample_sku]

fig = px.histogram(
    sample_data, x='daily_demand', 
    title=f"Distribución de Demanda Diaria - {sample_sku}",
    labels={'daily_demand': 'Demanda Diaria', 'count': 'Frecuencia'},
    nbins=20
)
fig.show()

📅 Demanda diaria agregada
Total registros: 6700


,sku,date,daily_demand
0,SKU-00001,2024-01-10,9
1,SKU-00001,2024-01-13,8
2,SKU-00001,2024-01-15,5
3,SKU-00001,2024-01-16,7
4,SKU-00001,2024-01-18,7


## 4️⃣ Cálculo de Variabilidad de Demanda

### 🎯 Caso de uso
- Cuantificar imprevisibilidad de demanda por SKU con `cv`.

### Métrica
- `cv = std / mean` (si mean > 0)

### Interpretación
- cv < 0.3: demanda estable
- 0.3 ≤ cv ≤ 0.7: demanda moderadamente variable
- cv > 0.7: alta variabilidad → considerar mayor safety stock

In [27]:
# Calcular estadísticas de demanda por SKU
demand_stats = df_daily_demand.groupby('sku')['daily_demand'].agg([
    ('avg_demand', 'mean'),
    ('std_demand', 'std'),
    ('cv', lambda x: x.std() / x.mean() if x.mean() > 0 else 0)  # Coeficiente de variación
]).reset_index()

# Enriquecer con información de producto
demand_stats = demand_stats.merge(df_products[['sku', 'category', 'brand']], on='sku')

print("📊 Estadísticas de Demanda:")
display(demand_stats.head())
print(f"\n📈 Promedio CV (variabilidad): {demand_stats['cv'].mean():.2f}")

📊 Estadísticas de Demanda:


,sku,avg_demand,std_demand,cv,category,brand
0,SKU-00001,12.933333,9.373232,0.724734,Household,BrandB
1,SKU-00002,11.888889,11.000721,0.925294,Electronics,BrandC
2,SKU-00003,10.909091,8.063667,0.739169,PersonalCare,BrandA
3,SKU-00004,14.604651,10.659535,0.729873,Electronics,BrandA
4,SKU-00005,14.024390,9.903251,0.706145,Electronics,BrandD



📈 Promedio CV (variabilidad): 0.78


## 5️⃣ Parámetros de Política de Inventario

In [28]:
# Parámetros de negocio
SERVICE_LEVEL = 0.95  # 95% nivel de servicio
LEAD_TIME_DAYS = 7    # 7 días lead time de reabastecimiento

# Z-score para nivel de servicio
z_score = stats.norm.ppf(SERVICE_LEVEL)

print(f"🎯 Parámetros de Política:")
print(f"  - Nivel de servicio: {SERVICE_LEVEL*100:.0f}%")
print(f"  - Z-score: {z_score:.2f}")
print(f"  - Lead time: {LEAD_TIME_DAYS} días")
print(f"\n📚 Interpretación: Con 95% de servicio, tenemos 5% de probabilidad de stockout")

🎯 Parámetros de Política:
  - Nivel de servicio: 95%
  - Z-score: 1.64
  - Lead time: 7 días

📚 Interpretación: Con 95% de servicio, tenemos 5% de probabilidad de stockout


## 6️⃣ Fórmula de Stock de Seguridad

### 🎯 Caso de uso
- Objetivo: Reducir probabilidad de ruptura de stock durante el lead time.
- Aplicación: Categorías con demanda aleatoria y lead time constante.

### Fórmula Clásica
```
Safety Stock = Z × σ_demanda × √(Lead Time)
```
Donde:
- **Z**: Z-score del nivel de servicio (e.g., 1.65 para 95%)
- **σ_demanda**: Desviación estándar diaria por SKU
- **Lead Time**: Días hasta reabastecer

### Interpretación de negocio
- Z ↑ → Más servicio → Más stock de seguridad → Menos rupturas
- σ ↑ → Demanda más volátil → Más stock de seguridad
- Lead Time ↑ → Más días cubiertos → Más stock de seguridad

### Buenas prácticas
- Usar ventanas móviles (p.ej., últimos 90 días) para σ
- Ajustar Z por criticidad de producto (A/B/C)
- Revisar nivel de servicio trimestral según fill rate real

In [29]:
# Calcular stock de seguridad
demand_stats['safety_stock'] = (
    z_score * demand_stats['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
).round(0).astype(int)

# Calcular punto de reorden (ROP = demanda durante lead time + safety stock)
demand_stats['reorder_point'] = (
    (demand_stats['avg_demand'] * LEAD_TIME_DAYS) + demand_stats['safety_stock']
).round(0).astype(int)

# Días de cobertura del safety stock
demand_stats['coverage_days'] = (
    demand_stats['safety_stock'] / demand_stats['avg_demand']
).round(1)

print("🛡️  Stock de Seguridad Calculado:")
display(demand_stats[[
    'sku', 'category', 'avg_demand', 'std_demand', 
    'safety_stock', 'reorder_point', 'coverage_days'
]])

🛡️  Stock de Seguridad Calculado:


,sku,category,avg_demand,std_demand,safety_stock,reorder_point,coverage_days
0,SKU-00001,Household,12.933333,9.373232,41,132,3.2
1,SKU-00002,Electronics,11.888889,11.000721,48,131,4.0
2,SKU-00003,PersonalCare,10.909091,8.063667,35,111,3.2
3,SKU-00004,Electronics,14.604651,10.659535,46,148,3.1
4,SKU-00005,Electronics,14.024390,9.903251,43,141,3.1
...,...,...,...,...,...,...,...
195,SKU-00196,PersonalCare,12.812500,11.882319,52,142,4.1
196,SKU-00197,PersonalCare,9.676471,8.303780,36,104,3.7
197,SKU-00198,Beverages,10.160000,6.780364,30,101,3.0
198,SKU-00199,Electronics,10.600000,7.452595,32,106,3.0


## 7️⃣ Análisis de Cobertura

### 🎯 Caso de uso
- Pregunta: ¿Cuántos días de demanda cubre el stock de seguridad?
- Decisión: Ajustar safety stock si cobertura es < 2 días (riesgo) o > 7 días (exceso capital).

### Métrica
- `coverage_days = safety_stock / avg_demand`

### Interpretación
- 2–4 días: razonable para retail con lead time semanal
- < 2 días: riesgo de ruptura ante picos
- > 7 días: capital inmovilizado; revisar política o lead time

In [30]:
# Top 10 productos con mayor safety stock
top_safety = demand_stats.nlargest(10, 'safety_stock')

fig = px.bar(
    top_safety,
    x='sku',
    y='safety_stock',
    color='category',
    title="Top 10 Productos por Stock de Seguridad",
    labels={'safety_stock': 'Safety Stock (unidades)', 'sku': 'SKU'}
)
fig.update_xaxes(tickangle=-45)
fig.show()

# Distribución de días de cobertura
fig2 = px.histogram(
    demand_stats, x='coverage_days',
    title="Distribución de Días de Cobertura del Safety Stock",
    labels={'coverage_days': 'Días de Cobertura'},
    nbins=20
)
fig2.show()

print(f"📊 Cobertura promedio: {demand_stats['coverage_days'].mean():.1f} días")
print(f"📊 Cobertura mediana: {demand_stats['coverage_days'].median():.1f} días")

📊 Cobertura promedio: 3.4 días
📊 Cobertura mediana: 3.4 días


## 8️⃣ Comparar con Inventario Actual

### 🎯 Caso de uso
- Detectar SKUs que necesitan reposición (stock actual < reorder point).
- Priorizar reabastecimiento por gap de inventario.

### Métricas
- `needs_replenishment`: booleano por SKU
- `stock_gap = reorder_point - current_stock` (clip a 0)

### Decisiones
- Gap alto → Orden inmediata
- Gap bajo → Consolidar en próxima orden
- Sin gap → Mantener niveles, monitorear

In [31]:
# Unir con inventario actual (on_hand por SKU)
inventory_comparison = demand_stats.merge(
    df_inventory.groupby('sku')['on_hand'].sum().reset_index(),
    on='sku',
    how='left'
)
inventory_comparison.rename(columns={'on_hand': 'current_stock'}, inplace=True)
inventory_comparison['current_stock'].fillna(0, inplace=True)

# Identificar productos con stock insuficiente
inventory_comparison['needs_replenishment'] = (
    inventory_comparison['current_stock'] < inventory_comparison['reorder_point']
)

# Gap de inventario
inventory_comparison['stock_gap'] = (
    inventory_comparison['reorder_point'] - inventory_comparison['current_stock']
).clip(lower=0)

print("🔍 Comparación con Inventario Actual:")
display(inventory_comparison[[
    'sku', 'category', 'current_stock', 'safety_stock', 
    'reorder_point', 'needs_replenishment', 'stock_gap'
]].head(10))

# Resumen
need_replen = inventory_comparison['needs_replenishment'].sum()
print(f"\n⚠️  Productos que necesitan reabastecimiento: {need_replen} de {len(inventory_comparison)}")
print(f"📦 Gap total de inventario: {inventory_comparison['stock_gap'].sum():.0f} unidades")

🔍 Comparación con Inventario Actual:


C:\Users\Luis\AppData\Local\Temp\ipykernel_25796\4284128028.py:8: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





,sku,category,current_stock,safety_stock,reorder_point,needs_replenishment,stock_gap
0,SKU-00001,Household,945,41,132,False,0
1,SKU-00002,Electronics,969,48,131,False,0
2,SKU-00003,PersonalCare,724,35,111,False,0
3,SKU-00004,Electronics,1380,46,148,False,0
4,SKU-00005,Electronics,732,43,141,False,0
5,SKU-00006,Snacks,769,34,118,False,0
6,SKU-00007,PersonalCare,708,43,128,False,0
7,SKU-00008,PersonalCare,987,37,112,False,0
8,SKU-00009,PersonalCare,524,22,83,False,0
9,SKU-00010,Electronics,1068,42,112,False,0



⚠️  Productos que necesitan reabastecimiento: 0 de 200
📦 Gap total de inventario: 0 unidades


## 9️⃣ Sensibilidad del Nivel de Servicio

### 🎯 Caso de uso
- Evaluar impacto de cambiar el nivel de servicio (90%, 95%, 98%, 99%).

### Interpretación
- A mayor `service_level` → mayor `z_score` → mayor `safety_stock`.
- Trade-off: menos stockouts vs mayor capital invertido.

### Recomendación
- Productos críticos (Clase A): 98%–99%
- Clase B: 95%
- Clase C: 90%–92%

In [32]:
# Analizar diferentes niveles de servicio
service_levels = [0.90, 0.95, 0.98, 0.99]
sample_sku_data = demand_stats.iloc[0]

sensitivity_results = []
for sl in service_levels:
    z = stats.norm.ppf(sl)
    ss = z * sample_sku_data['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
    sensitivity_results.append({
        'service_level': f"{sl*100:.0f}%",
        'z_score': z,
        'safety_stock': int(ss)
    })

df_sensitivity = pd.DataFrame(sensitivity_results)

fig = px.bar(
    df_sensitivity,
    x='service_level',
    y='safety_stock',
    title=f"Sensibilidad del Safety Stock al Nivel de Servicio<br>SKU: {sample_sku_data['sku']}",
    labels={'service_level': 'Nivel de Servicio', 'safety_stock': 'Safety Stock (unidades)'},
    text='safety_stock'
)
fig.update_traces(textposition='outside')
fig.show()

print("📊 Análisis de Sensibilidad:")
display(df_sensitivity)

📊 Análisis de Sensibilidad:


,service_level,z_score,safety_stock
0,90%,1.281552,31
1,95%,1.644854,40
2,98%,2.053749,50
3,99%,2.326348,57


## 🔟 Guardar Resultados

In [33]:
# Guardar políticas de inventario
output_file = OUTPUT_DIR / "inventory_policies.csv"
inventory_comparison.to_csv(output_file, index=False)

print(f"💾 Políticas guardadas: {output_file}")
print(f"📏 Dimensiones: {inventory_comparison.shape}")

# Resumen ejecutivo
summary = {
    'total_skus': len(inventory_comparison),
    'avg_safety_stock': inventory_comparison['safety_stock'].mean(),
    'total_safety_stock': inventory_comparison['safety_stock'].sum(),
    'skus_need_replenishment': need_replen,
    'total_stock_gap': inventory_comparison['stock_gap'].sum(),
    'service_level': SERVICE_LEVEL,
    'lead_time_days': LEAD_TIME_DAYS
}

print("\n📋 RESUMEN EJECUTIVO")
print("="*50)
for key, value in summary.items():
    print(f"  {key}: {value}")

💾 Políticas guardadas: ..\..\data\processed\inventory_policies.csv
📏 Dimensiones: (200, 12)

📋 RESUMEN EJECUTIVO
  total_skus: 200
  avg_safety_stock: 40.69
  total_safety_stock: 8138
  skus_need_replenishment: 0
  total_stock_gap: 0
  service_level: 0.95
  lead_time_days: 7


## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Variabilidad de Demanda**: La desviación estándar mide incertidumbre
2. ✅ **Nivel de Servicio**: Trade-off entre costo y servicio (95% → z=1.65)
3. ✅ **Fórmula Clásica**: `SS = Z × σ × √LT` captura incertidumbre
4. ✅ **Punto de Reorden**: `ROP = Demanda_LT + SS` dispara reabastecimiento

**Decisiones de Negocio:**
- 📊 Productos con alta variabilidad (CV > 0.5) requieren más safety stock
- 💰 Aumentar servicio de 95% a 99% incrementa SS en ~40%
- 🎯 Políticas diferenciadas por categoría ABC optimizan capital de trabajo

**Próximos Pasos:**
- Implementar políticas (Q, R) con tamaño de lote económico (EOQ)
- Incluir variabilidad de lead time (ver OR-01)
- Multi-echelon inventory optimization (ver OR-04)

---

**🔗 Notebooks Relacionados:**
- [OR-01: Stock de Seguridad](../50_optimization_or/OR-01-stock_seguridad.ipynb) - Versión avanzada
- [OR-02: Políticas de Inventario](../50_optimization_or/OR-02-politicas_inventario.ipynb)
- [BA-01: Dashboard OTIF](../40_business_analytics_bi/BA-01-dashboard_otif.ipynb)

## 🛠️ Funciones Reutilizables

In [34]:
def calculate_safety_stock(
    avg_demand: float,
    std_demand: float,
    service_level: float,
    lead_time_days: int
) -> dict:
    """
    Calcula stock de seguridad y punto de reorden.
    
    Args:
        avg_demand: Demanda promedio diaria
        std_demand: Desviación estándar de demanda diaria
        service_level: Nivel de servicio deseado (0-1)
        lead_time_days: Lead time de reabastecimiento (días)
    
    Returns:
        Dict con safety_stock, reorder_point, z_score
    """
    z_score = stats.norm.ppf(service_level)
    safety_stock = z_score * std_demand * np.sqrt(lead_time_days)
    reorder_point = (avg_demand * lead_time_days) + safety_stock
    
    return {
        'safety_stock': int(safety_stock),
        'reorder_point': int(reorder_point),
        'z_score': round(z_score, 2),
        'coverage_days': round(safety_stock / avg_demand, 1) if avg_demand > 0 else 0
    }

# Ejemplo de uso:
# result = calculate_safety_stock(avg_demand=50, std_demand=15, service_level=0.95, lead_time_days=7)
# print(result)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.

## 1️⃣2️⃣ Políticas por Categoría (LT/SL específicos)

### Objetivo
Ajustar `LEAD_TIME_DAYS` y `SERVICE_LEVEL` por categoría para reflejar casos reales.

### Configuración propuesta
- Electronics: SL 0.98, LT 10
- PersonalCare: SL 0.95, LT 7
- Household: SL 0.92, LT 5
- Snacks: SL 0.90, LT 4

Aplicaremos estos parámetros y recalcularemos safety stock y ROP por SKU.

In [35]:
# Políticas por Categoría: SL/LT específicos con tolerancia a datos faltantes
from scipy.stats import norm

# Parámetros por categoría (puedes ajustar según negocio)
category_params = {
    'Electronics': {'SERVICE_LEVEL': 0.98, 'LEAD_TIME_DAYS': 10},
    'PersonalCare': {'SERVICE_LEVEL': 0.95, 'LEAD_TIME_DAYS': 7},
    'Household': {'SERVICE_LEVEL': 0.92, 'LEAD_TIME_DAYS': 5},
    'Snacks': {'SERVICE_LEVEL': 0.90, 'LEAD_TIME_DAYS': 4},
    'Unknown': {'SERVICE_LEVEL': SERVICE_LEVEL, 'LEAD_TIME_DAYS': LEAD_TIME_DAYS},
}

# Asegurar que df_products tenga columna 'category'; si no, crear 'Unknown'
if 'category' not in df_products.columns:
    df_products = df_products.copy()
    df_products['category'] = 'Unknown'

# Función para calcular políticas por fila con tolerancia a categoría faltante
def compute_policy_row(row):
    cat = row.get('category', 'Unknown')
    params = category_params.get(cat, category_params['Unknown'])
    sl = params['SERVICE_LEVEL']
    lt = params['LEAD_TIME_DAYS']
    z = norm.ppf(sl)
    sigma = row['std_demand'] if pd.notnull(row['std_demand']) else 0.0
    avg = row['avg_demand'] if pd.notnull(row['avg_demand']) else 0.0
    safety_stock = z * sigma * (lt ** 0.5)
    reorder_point = avg * lt + safety_stock
    coverage_days = (row['current_stock'] / avg) if (avg and row.get('current_stock', 0) > 0) else 0.0
    return pd.Series({
        'service_level_cat': sl,
        'lead_time_days_cat': lt,
        'z_score_cat': z,
        'safety_stock_cat': safety_stock,
        'reorder_point_cat': reorder_point,
        'coverage_days_cat': coverage_days,
    })

# Preparar dataframe base y unir categoría
# Requiere que demand_stats esté definido y que inventory_stats tenga stock actual por sku
if 'current_stock' not in demand_stats.columns:
    # Unir stock actual si existe inventory_stats
    try:
        demand_stats = demand_stats.merge(inventory_stats[['sku','current_stock']], on='sku', how='left')
    except Exception:
        demand_stats['current_stock'] = 0

# Unir categoría desde df_products

demand_stats_cat = demand_stats.copy()
try:
    demand_stats_cat = demand_stats_cat.merge(df_products[['sku','category']], on='sku', how='left')
except KeyError:
    # Si sku no existe en df_products, continuar sin categoría
    demand_stats_cat['category'] = 'Unknown'

# Aplicar cálculo por fila de forma segura
computed = demand_stats_cat.apply(compute_policy_row, axis=1)

# Concatenar resultados
for col in computed.columns:
    demand_stats_cat[col] = computed[col]

print('📏 Políticas por categoría calculadas:')
try:
    display(demand_stats_cat[['sku','category','avg_demand','std_demand','service_level_cat','lead_time_days_cat','safety_stock_cat','reorder_point_cat','coverage_days_cat']].head())
except Exception:
    display(demand_stats_cat.head())

# Exportar CSV
output_dir = Path('../../data/processed/or07')
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'inventory_policies_by_category.csv'
demand_stats_cat.to_csv(output_file, index=False)
print(f'✅ Exportado: {output_file}')

📏 Políticas por categoría calculadas:


,sku,avg_demand,std_demand,cv,category_x,brand,safety_stock,reorder_point,coverage_days,current_stock,category_y,service_level_cat,lead_time_days_cat,z_score_cat,safety_stock_cat,reorder_point_cat,coverage_days_cat
0,SKU-00001,12.933333,9.373232,0.724734,Household,BrandB,41,132,3.2,0,Household,0.95,7.0,1.644854,40.791120,131.324453,0.0
1,SKU-00002,11.888889,11.000721,0.925294,Electronics,BrandC,48,131,4.0,0,Electronics,0.95,7.0,1.644854,47.873750,131.095972,0.0
2,SKU-00003,10.909091,8.063667,0.739169,PersonalCare,BrandA,35,111,3.2,0,PersonalCare,0.95,7.0,1.644854,35.092060,111.455697,0.0
3,SKU-00004,14.604651,10.659535,0.729873,Electronics,BrandA,46,148,3.1,0,Electronics,0.95,7.0,1.644854,46.388951,148.621509,0.0
4,SKU-00005,14.024390,9.903251,0.706145,Electronics,BrandD,43,141,3.1,0,Electronics,0.95,7.0,1.644854,43.097699,141.268431,0.0


✅ Exportado: ..\..\data\processed\or07\inventory_policies_by_category.csv


In [36]:
# Guardar artefactos de visualización (PNG/HTML) con tolerancia a dependencias
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# Usar el directorio de salida ya creado
out_dir = output_dir if 'output_dir' in globals() else Path('../../data/processed/or07')
out_dir.mkdir(parents=True, exist_ok=True)

# Figura: Top 10 Safety Stock
try:
    if 'top_safety' not in globals() or top_safety is None or top_safety.empty:
        top_safety = demand_stats.nlargest(10, 'safety_stock')
    # Determinar columna de categoría para color
    color_col = 'category_y' if 'category_y' in top_safety.columns else ('category_x' if 'category_x' in top_safety.columns else None)
    fig_top = px.bar(top_safety, x='sku', y='safety_stock', color=color_col, title='Top 10 Safety Stock')

    # Rutas
    png_top = out_dir / 'top10_safety_stock.png'
    html_top = out_dir / 'top10_safety_stock.html'

    # Intentar PNG; si falla (kaleido no instalado), guardar HTML
    try:
        pio.write_image(fig_top, png_top, format='png', width=1000, height=600)
        print(f'✅ PNG exportado: {png_top}')
    except Exception as e:
        print(f'⚠️ No se pudo exportar PNG (instala kaleido). Detalle: {e}')
    fig_top.write_html(html_top)
    print(f'✅ HTML exportado: {html_top}')
except Exception as e:
    print(f'❌ Error al generar Top 10 Safety Stock: {e}')

# Figura: Distribución de Cobertura en Días
try:
    if 'demand_stats_cat' in globals() and 'coverage_days_cat' in demand_stats_cat.columns:
        cov_df = demand_stats_cat[['sku','coverage_days_cat']].copy()
        fig_cov = px.histogram(cov_df, x='coverage_days_cat', nbins=20, title='Distribución de Cobertura en Días (por categoría)')
    else:
        cov_df = demand_stats[['sku','coverage_days']].copy() if 'coverage_days' in demand_stats.columns else demand_stats.copy()
        fig_cov = px.histogram(cov_df, x='coverage_days', nbins=20, title='Distribución de Cobertura en Días')

    png_cov = out_dir / 'coverage_days_distribution.png'
    html_cov = out_dir / 'coverage_days_distribution.html'

    try:
        pio.write_image(fig_cov, png_cov, format='png', width=1000, height=600)
        print(f'✅ PNG exportado: {png_cov}')
    except Exception as e:
        print(f'⚠️ No se pudo exportar PNG (instala kaleido). Detalle: {e}')
    fig_cov.write_html(html_cov)
    print(f'✅ HTML exportado: {html_cov}')
except Exception as e:
    print(f'❌ Error al generar distribución de cobertura: {e}')

✅ PNG exportado: ..\..\data\processed\or07\top10_safety_stock.png
✅ HTML exportado: ..\..\data\processed\or07\top10_safety_stock.html
✅ PNG exportado: ..\..\data\processed\or07\coverage_days_distribution.png
✅ HTML exportado: ..\..\data\processed\or07\coverage_days_distribution.html


## 1️⃣3️⃣ Resumen Ejecutivo (KPIs y Recomendaciones)

**KPIs:**
- Total SKUs: mostrar del dataset
- Safety Stock total (baseline vs por categoría)
- % SKUs que requieren reposición (gap>0)

**Recomendaciones:**
- Aumentar SL a 0.98 para Electronics si cobertura < 3 días
- Reducir SL a 0.92 para Household si cobertura > 7 días
- Revisar SKUs con cv > 0.8 (altamente variables)

Se guardaron artefactos en `../../data/processed/or07/` (CSV/PNG/HTML).

---## 📚 Navegación**[📑 Índice del Proyecto](..\..\README.md)** | **[📋 Catálogo de Notebooks](..\..\config\notebooks_index.yml)**← Anterior: [OR-06](..\50_optimization_or\OR-06-dock_queue_simulation.ipynb) | Siguiente: [OR-08](..\50_optimization_or\OR-08-production_scheduling.ipynb) →